# 🧠 Lab 4: LLM & Multimodal LLM Practice
## Text Generation (LLM) → Vision-Language Understanding (Qwen3-VL)

This is the final practice that brings together everything from Labs 1~3.

### Part A — LLM: How text generation works
1. How a Causal LM works — visualize next-token prediction
2. Compare temperature & sampling strategies
3. Prompt engineering practice — generate Dog/Cat-related text

### Part B — Multimodal LLM: Qwen3-VL
4. Load the Qwen3-VL model
5. Image captioning — experiment on the same Pet data
6. Visual Question Answering (VQA)
7. Image comparison & difference analysis
8. Structured output (JSON) generation
9. Multi-turn conversation (image-grounded follow-ups)
10. Lab 1~4 Grand Summary

### Foundation Model Evolution
```
Lab 1: DINO ViT          — image feature learning (self-supervised)
Lab 2: Grounding DINO+SAM — find and cut with text (detection+segmentation)
Lab 3: BERT + CLIP        — text understanding + multimodal alignment (contrastive)
Lab 4: LLM + Qwen3-VL     — text generation + 'understanding' images and chatting (generative)
```

---

## 0. Environment Setup

In [ ]:
# Qwen3-VL requires transformers >= 4.57.0
!pip install "transformers>=4.57.0" torch torchvision accelerate Pillow matplotlib numpy tqdm requests qwen-vl-utils -q

In [ ]:
import os
import random
import tarfile
import requests
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,   # LLM
    Qwen3VLForConditionalGeneration,        # Multimodal LLM
    AutoProcessor,
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Data Preparation (same as Labs 1~3)

In [ ]:
NUM_IMAGES_PER_CLASS = 50
IMG_DIR = './data'
CACHE_DIR = './cache'
for d in [IMG_DIR, CACHE_DIR, f'{IMG_DIR}/dog', f'{IMG_DIR}/cat']:
    os.makedirs(d, exist_ok=True)

def download_oxford_pets():
    extract_dir = os.path.join(CACHE_DIR, 'images')
    if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 100:
        print('   Using cache'); return extract_dir
    url = 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz'
    tar_path = os.path.join(CACHE_DIR, 'images.tar.gz')
    print(f'   Downloading: {url}')
    r = requests.get(url, stream=True, timeout=120)
    with open(tar_path, 'wb') as f:
        for c in r.iter_content(1024*1024): f.write(c)
    with tarfile.open(tar_path, 'r:gz') as t: t.extractall(CACHE_DIR)
    return os.path.join(CACHE_DIR, 'images')

def organize(src, dst, n):
    dogs = sorted([f for f in os.listdir(src) if f[0].islower() and f.endswith(('.jpg','.jpeg','.png'))])
    cats = sorted([f for f in os.listdir(src) if f[0].isupper() and f.endswith(('.jpg','.jpeg','.png'))])
    random.shuffle(dogs); random.shuffle(cats)
    for i,f in enumerate(dogs[:n]): Image.open(f'{src}/{f}').convert('RGB').save(f'{dst}/dog/dog_{i:03d}.jpg')
    for i,f in enumerate(cats[:n]): Image.open(f'{src}/{f}').convert('RGB').save(f'{dst}/cat/cat_{i:03d}.jpg')

ed = len([f for f in os.listdir(f'{IMG_DIR}/dog') if f.endswith('.jpg')])
ec = len([f for f in os.listdir(f'{IMG_DIR}/cat') if f.endswith('.jpg')])
if ed >= NUM_IMAGES_PER_CLASS and ec >= NUM_IMAGES_PER_CLASS:
    print(f'✅ Data ready: Dogs={ed}, Cats={ec}')
else:
    organize(download_oxford_pets(), IMG_DIR, NUM_IMAGES_PER_CLASS)

image_paths, image_labels = [], []
for label in ['dog', 'cat']:
    for f in sorted(os.listdir(f'{IMG_DIR}/{label}')):
        if f.endswith('.jpg'): image_paths.append(f'{IMG_DIR}/{label}/{f}'); image_labels.append(label)
print(f"📂 Loaded {len(image_paths)} images")

---
# Part A: LLM (Causal Language Model)

## 2. Visualize Next-Token Prediction

The core of an LLM: it repeatedly **predicts the next token given the previous tokens**.

```
"The dog"  →  P(next) = {"is": 0.3, "was": 0.15, "runs": 0.1, ...}
"The dog is"  →  P(next) = {"a": 0.2, "very": 0.15, "cute": 0.1, ...}
```

While BERT (Lab 3) understood context bidirectionally,
an LLM generates the next word looking only **left → right** in one direction.

In [ ]:
# Demonstrate the next-token prediction principle with a small GPT-2 model
from transformers import GPT2LMHeadModel, GPT2Tokenizer

LLM_ID = 'gpt2'  # 124M — light and fast, for principle demonstration
print(f"📦 Loading LLM: {LLM_ID}")
llm_tokenizer = GPT2Tokenizer.from_pretrained(LLM_ID)
llm_model = GPT2LMHeadModel.from_pretrained(LLM_ID).to(device)
llm_model.eval()

params = sum(p.numel() for p in llm_model.parameters())
print(f"   ✅ Loaded! ({params/1e6:.0f}M params)")
print(f"   - Vocab size: {llm_tokenizer.vocab_size}")
print(f"   - Max length: {llm_model.config.n_positions}")

In [ ]:
@torch.no_grad()
def visualize_next_token(text, top_k=10):
    """Visualize the probability distribution of the next token."""
    inputs = llm_tokenizer(text, return_tensors='pt').to(device)
    outputs = llm_model(**inputs)
    
    # Logits at the last position → next-token probabilities
    next_logits = outputs.logits[0, -1, :]
    probs = torch.softmax(next_logits, dim=0)
    top_probs, top_ids = probs.topk(top_k)
    
    tokens = [llm_tokenizer.decode(idx.item()) for idx in top_ids]
    probs_list = top_probs.cpu().numpy()
    
    return tokens, probs_list

# Predict next token for several prompts
prompts = [
    "The dog is",
    "A cute little cat",
    "Dogs and cats are popular",
    "The golden retriever",
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('🔮 LLM Next Token Prediction — "What word comes next?"', fontsize=16, fontweight='bold')

colors = ['#4ECDC4', '#FF6B6B', '#45B7D1', '#96CEB4']

for i, prompt in enumerate(prompts):
    ax = axes[i // 2, i % 2]
    tokens, probs = visualize_next_token(prompt)
    
    # Clean up tokens for display (whitespace/special chars)
    display_tokens = [repr(t).strip("'") for t in tokens]
    
    bars = ax.barh(range(len(tokens)), probs, color=colors[i], alpha=0.8, edgecolor='black')
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(display_tokens, fontsize=10, fontfamily='monospace')
    ax.set_xlabel('Probability')
    ax.set_title(f'"{prompt}" + ???', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    
    for bar, p in zip(bars, probs):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
               f'{p:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 3. Temperature & Sampling Strategies

There are several ways an LLM can pick the next token:

| Strategy | Description |
|------|------|
| **Greedy** | Always pick the most probable token (repetitive) |
| **Temperature** | Smooth/sharpen the probability distribution |
| **Top-k** | Sample only from the top-k candidates |
| **Top-p (nucleus)** | Sample from candidates up to cumulative probability p |

In [ ]:
def generate_text(prompt, max_new_tokens=60, temperature=1.0, top_k=50, top_p=0.9, do_sample=True):
    """Generate text with GPT-2."""
    inputs = llm_tokenizer(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    
    generated = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated

# Compare temperatures
prompt = "Dogs are wonderful pets because"

print("🌡️ Temperature comparison\n")
print(f"  Prompt: \"{prompt}\"\n")

temperatures = [0.3, 0.7, 1.0, 1.5]
for temp in temperatures:
    result = generate_text(prompt, temperature=temp, max_new_tokens=50)
    new_text = result[len(prompt):].strip()
    print(f"  🌡️ T={temp}: {new_text[:120]}...")
    print()

print("💡 Low temperature → conservative and repetitive")
print("   High temperature → creative but can be off-topic")

In [ ]:
# Visualize how temperature changes the probability distribution
prompt = "The cat sat on the"
inputs = llm_tokenizer(prompt, return_tensors='pt').to(device)
with torch.no_grad():
    logits = llm_model(**inputs).logits[0, -1, :]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle(f'🌡️ Effect of temperature on the probability distribution — "{prompt}" + ???',
            fontsize=14, fontweight='bold')

for i, temp in enumerate([0.1, 0.5, 1.0, 2.0]):
    probs = torch.softmax(logits / temp, dim=0)
    top_probs, top_ids = probs.topk(8)
    tokens = [repr(llm_tokenizer.decode(idx.item())).strip("'") for idx in top_ids]
    
    axes[i].bar(range(len(tokens)), top_probs.cpu().numpy(), color=plt.cm.coolwarm(temp/2.5),
               edgecolor='black', alpha=0.8)
    axes[i].set_xticks(range(len(tokens)))
    axes[i].set_xticklabels(tokens, rotation=45, ha='right', fontsize=9, fontfamily='monospace')
    axes[i].set_ylabel('Probability')
    axes[i].set_ylim(0, 1)
    axes[i].set_title(f'T = {temp}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 T=0.1: nearly greedy (mass concentrates on the top one) / T=2.0: nearly uniform (spread evenly)")

In [ ]:
# Generate Dog/Cat-related text (various prompts)
creative_prompts = [
    "A day in the life of a golden retriever:",
    "Why cats are the best pets:",
    "The difference between dogs and cats is that",
    "If my dog could talk, it would say:",
]

print("📝 LLM text generation (Dog/Cat topics)\n")
for p in creative_prompts:
    result = generate_text(p, temperature=0.8, max_new_tokens=60)
    print(f"  💬 Prompt: \"{p}\"")
    print(f"  📖 Generated: {result[:200]}")
    print()

---
# Part B: Multimodal LLM (Qwen3-VL)

## 4. Load the Qwen3-VL Model

**Qwen3-VL** (Qwen Team, 2025) is a state-of-the-art Multimodal LLM:
- Integrates a Vision Transformer + Language Model
- Can "see" images and "describe" / "chat" about them in natural language
- The 2B model offers enough performance for practice with reasonable VRAM usage

```
Image ──→ [ViT Encoder] ──→ Visual Tokens ─┐
                                              ├──→ [LLM Decoder] ──→ Text Output
Text  ──→ [Tokenizer]   ──→ Text Tokens   ─┘
```

> ⚠️ **GPU required**: Qwen3-VL-2B needs about **5~8GB VRAM**.
> It can run on CPU, but it is very slow.

In [ ]:
VLM_ID = "Qwen/Qwen3-VL-2B-Instruct"

print(f"📦 Loading Qwen3-VL: {VLM_ID}")
print(f"   (the model download takes a few minutes on first run)\n")

vlm_processor = AutoProcessor.from_pretrained(VLM_ID)

vlm_model = Qwen3VLForConditionalGeneration.from_pretrained(
    VLM_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

if not torch.cuda.is_available():
    vlm_model = vlm_model.to(device)

vlm_model.eval()

vlm_params = sum(p.numel() for p in vlm_model.parameters())
print(f"   ✅ Loaded!")
print(f"   - Parameters: {vlm_params / 1e9:.1f}B")
print(f"   - dtype: {next(vlm_model.parameters()).dtype}")

In [ ]:
from qwen_vl_utils import process_vision_info

def ask_vlm(image_path, question, max_new_tokens=256, temperature=0.7):
    """
    Send an image and a question to Qwen3-VL and get the response.
    
    Args:
        image_path: str, path to the image file
        question: str, the question text
    Returns:
        str: model response
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
                {"type": "text", "text": question},
            ],
        }
    ]
    
    text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = vlm_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(vlm_model.device)
    
    with torch.no_grad():
        generated_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
        )
    
    # Strip the input portion
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    response = vlm_processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
    
    return response.strip()

print("✅ ask_vlm function defined")

---
## 5. Image Captioning — Generating Image Descriptions

We show Qwen3-VL the same Oxford-IIIT Pet images and request descriptions.

In [ ]:
# Generate captions for Dog/Cat images
print("📸 Qwen3-VL Image Captioning\n")

caption_samples = [0, 2, NUM_IMAGES_PER_CLASS, NUM_IMAGES_PER_CLASS + 2]

fig, axes = plt.subplots(1, len(caption_samples), figsize=(5 * len(caption_samples), 6))
fig.suptitle('📸 Qwen3-VL — Image Captioning', fontsize=16, fontweight='bold')

for i, idx in enumerate(caption_samples):
    img = Image.open(image_paths[idx]).convert('RGB')
    gt = image_labels[idx].upper()
    
    caption = ask_vlm(image_paths[idx], "Describe this image in one detailed sentence.")
    
    axes[i].imshow(img)
    # Wrap long captions
    wrapped = '\n'.join([caption[j:j+35] for j in range(0, min(len(caption), 140), 35)])
    axes[i].set_title(f'[{gt}]\n{wrapped}', fontsize=9, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

---
## 6. Visual Question Answering (VQA)

We ask various questions about an image.

In [ ]:
# Various questions about a single image
sample_path = image_paths[0]
sample_img = Image.open(sample_path).convert('RGB')
sample_gt = image_labels[0]

questions = [
    "What animal is in this image?",
    "What breed do you think this is?",
    "Describe the background of this image.",
    "What emotion does this animal seem to be expressing?",
    "Is this animal indoors or outdoors?",
    "Estimate the age of this animal (puppy/adult/senior).",
]

print(f"🐾 VQA experiment — {sample_gt.upper()} image\n")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(sample_img)
ax.set_title(f'Test Image ({sample_gt.upper()})', fontsize=12, fontweight='bold')
ax.axis('off')
plt.show()

print("\n❓ Questions & answers:\n")
for q in questions:
    answer = ask_vlm(sample_path, q, max_new_tokens=100)
    print(f"  Q: {q}")
    print(f"  A: {answer}")
    print()

---
## 7. Image Comparison — Dog vs Cat

In [ ]:
def ask_vlm_multi_image(image_paths_list, question, max_new_tokens=300):
    """Show multiple images at once and ask a question."""
    content = []
    for i, path in enumerate(image_paths_list):
        content.append({"type": "image", "image": f"file://{os.path.abspath(path)}"})
        content.append({"type": "text", "text": f"(Image {i+1})"})
    content.append({"type": "text", "text": question})
    
    messages = [{"role": "user", "content": content}]
    
    text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vlm_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(vlm_model.device)
    
    with torch.no_grad():
        gen_ids = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
    return vlm_processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

# Compare a Dog vs Cat image
dog_path = image_paths[0]
cat_path = image_paths[NUM_IMAGES_PER_CLASS]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(dog_path)); axes[0].set_title('Image 1 (Dog)', fontweight='bold'); axes[0].axis('off')
axes[1].imshow(Image.open(cat_path)); axes[1].set_title('Image 2 (Cat)', fontweight='bold'); axes[1].axis('off')
plt.suptitle('🔍 Image comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

comparison = ask_vlm_multi_image(
    [dog_path, cat_path],
    "Compare these two animals. What are the differences and similarities between Image 1 and Image 2?"
)
print(f"\n🔍 Comparison result:\n{comparison}")

---
## 8. Structured Output (JSON) Generation

If we ask the multimodal LLM to respond in **JSON format**,
we get structured data we can use directly in code.

In [ ]:
json_prompt = """Analyze this image and respond ONLY with a JSON object (no markdown, no explanation) with these fields:
- "animal": the type of animal ("dog" or "cat")
- "breed_guess": your best guess of the breed
- "color": main color of the animal
- "setting": "indoor" or "outdoor"
- "mood": the apparent mood of the animal
- "confidence": your confidence in the animal type (0-100)"""

print("📋 Generating structured output (JSON)\n")

json_samples = [0, 1, NUM_IMAGES_PER_CLASS, NUM_IMAGES_PER_CLASS + 1]

fig, axes = plt.subplots(1, len(json_samples), figsize=(5 * len(json_samples), 5))

for i, idx in enumerate(json_samples):
    img = Image.open(image_paths[idx]).convert('RGB')
    gt = image_labels[idx].upper()
    
    response = ask_vlm(image_paths[idx], json_prompt, max_new_tokens=200, temperature=0.1)
    
    axes[i].imshow(img)
    axes[i].set_title(f'GT: {gt}', fontsize=11, fontweight='bold')
    axes[i].axis('off')
    
    print(f"  [{gt}] {os.path.basename(image_paths[idx])}:")
    
    # Try to parse JSON
    try:
        # Extract only the JSON part
        json_str = response
        if '{' in json_str:
            json_str = json_str[json_str.index('{'):json_str.rindex('}')+1]
        parsed = json.loads(json_str)
        print(f"  {json.dumps(parsed, indent=4)}")
    except:
        print(f"  (Raw) {response[:200]}")
    print()

plt.tight_layout()
plt.show()

---
## 9. Multi-Turn Conversation (image-grounded follow-ups)

A strength of multimodal LLMs: you can have **multiple turns of conversation** about an image.

In [ ]:
def multi_turn_conversation(image_path, conversation_questions):
    """
    Run a multi-turn conversation about an image.
    """
    messages = []
    results = []
    
    for i, question in enumerate(conversation_questions):
        if i == 0:
            # Include the image only in the first question
            messages.append({
                "role": "user",
                "content": [
                    {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
                    {"type": "text", "text": question},
                ],
            })
        else:
            messages.append({
                "role": "user",
                "content": [{"type": "text", "text": question}],
            })
        
        text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = vlm_processor(
            text=[text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt"
        ).to(vlm_model.device)
        
        with torch.no_grad():
            gen_ids = vlm_model.generate(**inputs, max_new_tokens=200)
        
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
        response = vlm_processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
        
        # Add the response to the conversation history
        messages.append({"role": "assistant", "content": [{"type": "text", "text": response}]})
        results.append((question, response))
    
    return results

# Run a multi-turn conversation
sample_path = image_paths[0]

conversation = [
    "What do you see in this image?",
    "What breed do you think it is and why?",
    "What kind of food would be best for this animal?",
    "Write a short funny caption for this image.",
]

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(Image.open(sample_path))
ax.set_title('Conversation Subject', fontweight='bold')
ax.axis('off')
plt.show()

print("\n💬 Multi-turn conversation:\n")
results = multi_turn_conversation(sample_path, conversation)
for i, (q, a) in enumerate(results):
    print(f"  👤 Turn {i+1}: {q}")
    print(f"  🤖 {a}")
    print()

---
## 10. Batch Classification — Dog/Cat Classification with the VLM

Compared to CLIP zero-shot in Lab 3: can the VLM also identify Dog/Cat from an image?

In [ ]:
# Run a VLM classification experiment on a subset of images
n_test = min(20, len(image_paths))  # VLM is slow, so only a subset
test_indices = random.sample(range(len(image_paths)), n_test)

classify_prompt = 'Is this a dog or a cat? Answer with only one word: "dog" or "cat".'

print(f"🏷️ VLM Classification ({n_test} images)\n")

vlm_preds = []
vlm_gts = []

for idx in tqdm(test_indices, desc='Classifying'):
    response = ask_vlm(image_paths[idx], classify_prompt, max_new_tokens=10, temperature=0.1)
    pred = 'dog' if 'dog' in response.lower() else 'cat' if 'cat' in response.lower() else 'unknown'
    vlm_preds.append(pred)
    vlm_gts.append(image_labels[idx])

# Compute accuracy
correct = sum(1 for p, g in zip(vlm_preds, vlm_gts) if p == g)
total = len(vlm_gts)
acc = correct / total

print(f"\n{'='*50}")
print(f"  🏷️ VLM Classification Accuracy: {acc:.1%} ({correct}/{total})")
print(f"{'='*50}")

# Visualize: a subset of the results
n_show = min(8, n_test)
fig, axes = plt.subplots(2, n_show // 2, figsize=(4 * (n_show // 2), 8))
fig.suptitle(f'🏷️ Qwen3-VL Classification (Acc: {acc:.1%})', fontsize=14, fontweight='bold')

for i in range(n_show):
    ax = axes[i // (n_show // 2), i % (n_show // 2)]
    idx = test_indices[i]
    img = Image.open(image_paths[idx]).convert('RGB')
    ax.imshow(img)
    
    gt, pred = vlm_gts[i], vlm_preds[i]
    ok = gt == pred
    ax.set_title(f"{'✓' if ok else '✗'} GT:{gt.upper()} Pred:{pred.upper()}",
                color='green' if ok else 'red', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 📝 Lab 1~4 Grand Summary

### Per-Model Comparison

| Lab | Model | Paradigm | Input | Output | Core ability |
|-----|------|---------|------|------|----------|
| 1 | DINO ViT | Self-supervised | Image | Feature vector | Feature learning, attention |
| 2 | Grounding DINO + SAM | Open-set detection | Image + Text | Boxes + Masks | Detection, segmentation |
| 3A | BERT | Masked LM | Text | Feature / token prediction | Text understanding |
| 3B | CLIP | Contrastive | Image + Text | Aligned embeddings | Zero-shot, retrieval |
| 4A | GPT-2 | Causal LM | Text | Text | Text generation |
| 4B | Qwen3-VL | Multimodal LLM | Image + Text | Text | VQA, captioning, reasoning |

### Foundation Model Evolution Timeline

```
2018  BERT        — bidirectional text understanding (Masked LM)
2019  GPT-2       — unidirectional text generation (Causal LM)
2021  DINO        — image self-supervised learning (Self-distillation)
2021  CLIP        — image-text alignment (Contrastive)
2023  SAM         — general-purpose segmentation (foundation model)
2023  Grounding DINO — text-driven object detection (open-vocabulary)
2023  GPT-4V      — start of multimodal LLMs
2025  Qwen3-VL    — state-of-the-art open-source multimodal LLM
```

### DINO vs CLIP vs VLM — Same image, different ways of understanding

| | DINO (Lab 1) | CLIP (Lab 3) | Qwen3-VL (Lab 4) |
|---|---|---|---|
| Way of understanding | Visual feature extraction | Image-text similarity | Look at image and converse |
| Classification | Requires linear probing | Zero-shot (text comparison) | Direct answer generation |
| Explainability | Attention map | Similarity score | Natural-language explanation |
| Flexibility | Fixed task | Can change text | Any question is possible |

### Key Takeaways

1. **Self-supervised learning** (DINO, BERT) → strong representations can be learned without labels
2. **Contrastive learning** (CLIP) → aligns different modalities into a single space
3. **Generative models** (GPT, Qwen3-VL) → go beyond understanding to generation and reasoning
4. **Multimodal LLM** → the ultimate interface that integrates all modalities and communicates in natural language

### Things to explore further
- Use Qwen3-VL to do bounding-box / grounding instructions
- Build an automatic data labeling pipeline using a VLM
- Practice VLM fine-tuning with LoRA/QLoRA
- RAG (Retrieval Augmented Generation) + VLM

---
*This notebook is the LLM & Multimodal LLM practice (Lab 4).*
*The Lab 1~4 series covers all the core concepts of foundation models!*